# USDA Cropland Data Layer (CDL) — Crop History for Every Field

**What it is.** An annual, crop-specific satellite land-cover map of the continental U.S.
For any point you can read the **crop grown each year** — i.e. reconstruct a field's
**rotation history** — and for any county get **acreage by crop**.

| | |
|---|---|
| **Coverage** | Continental U.S. |
| **History** | Nationwide 2008–present (10 m from 2024; 30 m before) |
| **Cost / key** | **Free · no key** (CropScape / CroplandCROS, hosted at GMU) |
| **Docs** | https://nassgeodata.gmu.edu/CropScape/ · https://croplandcros.scinet.usda.gov/ |

**Why it's core to Tracera.** Rotation history predicts pest/disease pressure, nitrogen
credits, and expected yield. This notebook loops over **every field** defined in
`data_sources/_common.py` (add a field there and it shows up here automatically).

In [1]:
# Tracera setup — locate the repo root so `data_sources._common` imports no matter
# where the notebook is launched from. Your repo is nested (...\Tracera\Tracera),
# so we can't assume the working directory.
import sys, pathlib, re, io, requests, pandas as pd

def _add_repo_to_path():
    starts = []
    nb = globals().get("__vsc_ipynb_file__")   # VS Code exposes the notebook's path
    if nb:
        starts.append(pathlib.Path(nb).resolve().parent)
    starts.append(pathlib.Path.cwd())
    for base in starts:
        for p in [base, *base.parents]:
            if (p / "data_sources" / "_common.py").exists():
                if str(p) not in sys.path:
                    sys.path.insert(0, str(p))
                return p
    raise RuntimeError("Could not find data_sources/_common.py above the notebook or CWD.")

REPO_ROOT = _add_repo_to_path()
from data_sources._common import FIELDS, get_key, field_polygon

# FIELDS is a LIST of field dicts. Loop over it for multi-field, or grab one:
#   FIELDS[0]["lat"]  ->  a single value    (NOT FIELDS["lat"] — that's a TypeError)
PRIMARY = FIELDS[0]
LAT, LON = PRIMARY["lat"], PRIMARY["lon"]
FIPS, STATE, COUNTY = PRIMARY["fips"], PRIMARY["state_alpha"], PRIMARY["county_name"]

print(f"Repo root: {REPO_ROOT}")
print(f"{len(FIELDS)} field(s) loaded:")
for f in FIELDS:
    print(f"  {f['id']} — {f['name']}: {f['acres']} ac {f['crop']} | "
          f"{f['county_name']} Co, {f['state_alpha']} | ({f['lat']}, {f['lon']})")

Repo root: c:\Users\kylep\source\repos\kyleparran\Tracera\Tracera
2 field(s) loaded:
  Ault's — North 80: 20.49 ac corn | MONROE Co, MI | (41.896389, -83.62375)
  Rado's — Field 2: 67 ac soybeans | MONROE Co, MI | (41.909583, -83.632444)


### 1. Connection test — the CDL crop at each field (latest full year)

In [2]:
from pyproj import Transformer
import time

CDL = "https://nassgeodata.gmu.edu/axis2/services/CDLService"
# CDL rasters are in Albers Equal Area (EPSG:5070); GetCDLValue needs x/y in metres.
_to_albers = Transformer.from_crs("EPSG:4326", "EPSG:5070", always_xy=True)

def _cdl_get(path, params, timeout=60, tries=3):
    """GET the GMU CropScape service with retries (the public service can be slow)."""
    for i in range(tries):
        try:
            r = requests.get(f"{CDL}/{path}", params=params, timeout=timeout)
            r.raise_for_status()
            return r
        except requests.exceptions.RequestException:
            if i == tries - 1:
                raise
            time.sleep(3 * (i + 1))

def cdl_crop(year, lat, lon):
    """CDL crop class at a point for a year (None if the service is unavailable/slow)."""
    x, y = _to_albers.transform(lon, lat)
    try:
        r = _cdl_get("GetCDLValue", {"year": year, "x": x, "y": y})
    except requests.exceptions.RequestException:
        return None
    m = re.search(r'value: (\d+), category: "([^"]+)"', r.text)
    return m.group(2) if m else None

for f in FIELDS:
    print(f"{f['id']} — {f['name']}: {cdl_crop(2024, f['lat'], f['lon'])} (2024 CDL)")

Ault's — North 80: Corn (2024 CDL)


Rado's — Field 2: Soybeans (2024 CDL)


### 2. Crop rotation history — one column per field

In [3]:
YEARS = range(2015, 2025)   # 2024 = latest CDL (10 m); extend as new years release
rotation = pd.DataFrame(
    {f"{f['id']} — {f['name']}": {y: cdl_crop(y, f['lat'], f['lon']) for y in YEARS}
     for f in FIELDS}
)
rotation.index.name = "year"
print("Crop rotation at each field:")
rotation

Crop rotation at each field:


,Ault's — North 80,Rado's — Field 2
year,,
2015,Corn,Corn
2016,Soybeans,Soybeans
2017,Soybeans,Corn
2018,Corn,Soybeans
2019,Soybeans,Corn
2020,Corn,Soybeans
2021,Soybeans,Corn
2022,Corn,Soybeans
2023,Soybeans,Corn


### 3. County crop acreage (per county, deduplicated)

In [4]:
def cdl_county_acreage(year, fips):
    r = _cdl_get("GetCDLStat", {"year": year, "fips": fips, "format": "csv"}, timeout=120)
    url = re.search(r"<returnURL>(.*?)</returnURL>", r.text).group(1)
    df = pd.read_csv(io.StringIO(requests.get(url, timeout=60).text))
    df.columns = [c.strip() for c in df.columns]
    return df.sort_values("Acreage", ascending=False)

shown = set()
for f in FIELDS:
    if f["fips"] in shown:
        continue
    shown.add(f["fips"])
    acres = cdl_county_acreage(2023, f["fips"])
    print(f"{f['county_name']} County, {f['state_alpha']} ({f['fips']}) — top crops, 2023:")
    display(acres.head(8))

MONROE County, MI (26115) — top crops, 2023:


,Value,Category,Count,Acreage
2,5,Soybeans,396533,88186.8
0,1,Corn,274546,61057.6
36,141,Deciduous Forest,200968,44694.2
40,176,Grass/Pasture,146385,32555.2
6,24,Winter Wheat,116379,25882.1
32,122,Developed/Low Intensity,112129,24936.9
31,121,Developed/Open Space,107737,23960.1
33,123,Developed/Medium Intensity,58473,13004.1


### Notes & how to extend
- **Add fields** in `data_sources/_common.py` (the `FIELDS` list) — every cell here loops over
  them automatically. Each field carries its own county `fips`, so multi-county farms work too.
- **Category codes:** 1 = Corn, 5 = Soybeans, 24 = Winter Wheat, 176 = Grass/Pasture, etc.
- **Nitrogen credit:** a field showing Soybeans last year earns an N credit for corn this year —
  a signal the field report can compute from `rotation`.
- **Field polygons:** `GetCDLFile(year, fips|bbox)` returns a clipped GeoTIFF you can zonal-stat
  against a real field boundary for a true per-field (not single-point) rotation.
- **Rate limits:** the GMU service is public and can be slow — cache results to `data_cache/`.